In [3]:
import sage.all as sage
import numpy as np
import itertools
from sage.rings.rational_field import QQ
from sage.combinat.sf.sf import SymmetricFunctions
import math


_Sym = SymmetricFunctions(QQ)
_s   = _Sym.schur()
_p   = _Sym.powersum()

Get all $\pi_\lambda \in S_k$ 

In [4]:
def conjugacy_classes_and_character(k: int, lam: list[int]) -> list[tuple]:
    assert sum(lam) == k
    lam = sorted(lam, reverse=True)  # normalize to non-increasing partition

    Sk = sage.SymmetricGroup(k)
    ct = Sk.character_table()
    classes = Sk.conjugacy_classes()

    partitions_ordered = list(reversed(sage.Partitions(k).list()))
    row_idx = partitions_ordered.index(lam)

    return [(cl.list(), ct[row_idx][j]) for j, cl in enumerate(classes)]

# Should match known character table for S_3
for lam in sage.Partitions(3).list():
    result = conjugacy_classes_and_character(3, lam)
    print(f"\nlambda = {lam}:")
    for elements, chi in result:
        print(f"  sigma = {elements[0]}, chi = {chi}")


lambda = [3]:
  sigma = (), chi = 1
  sigma = (2,3), chi = 1
  sigma = (1,2,3), chi = 1

lambda = [2, 1]:
  sigma = (), chi = 2
  sigma = (2,3), chi = 0
  sigma = (1,2,3), chi = -1

lambda = [1, 1, 1]:
  sigma = (), chi = 1
  sigma = (2,3), chi = -1
  sigma = (1,2,3), chi = 1


In [5]:
from itertools import product as iproduct
import numpy as np

def sigma_matrix_on_tensor_product(sigma, n, k):
    """
    Matrix of sigma in S_k acting on V^{otimes k}, dim(V) = n.
    sigma: element of Sage's SymmetricGroup(k)
    """
    basis = list(iproduct(range(n), repeat=k))
    index = {b: i for i, b in enumerate(basis)}
    
    sigma_inv = sigma.inverse()
    dim = n**k
    M = np.zeros((dim, dim), dtype=complex)
    
    for col_idx, basis_vec in enumerate(basis):
        # permute indices by sigma_inv
        new_basis = tuple(basis_vec[sigma_inv(j + 1) - 1] for j in range(k))
        row_idx = index[new_basis]
        M[row_idx, col_idx] = 1.0
    
    return M

In [6]:
def isotypic_projector(lam: list[int], n: int, k: int) -> np.ndarray:
    """
    Compute Pi_lambda on V^{otimes k} where dim(V) = n.
    Uses the character table for efficiency.
    """
    assert sum(lam) == k
    lam = sorted(lam, reverse=True)

    Sk = sage.SymmetricGroup(k)
    ct = Sk.character_table()
    classes = Sk.conjugacy_classes()

    # Match lam to character table row
    partitions_ordered = list(reversed(sage.Partitions(k).list()))
    row_idx = partitions_ordered.index(lam)

    # dimension d_lambda = chi_lambda(identity)
    id_idx = next(j for j, cl in enumerate(classes)
                  if cl.representative().is_one())
    d_lam = int(ct[row_idx][id_idx])

    dim = n**k
    Pi = np.zeros((dim, dim), dtype=complex)

    for j, cl in enumerate(classes):
        chi = complex(ct[row_idx][j])
        # all elements in same class share the same character
        # so compute matrix per element, multiply by chi
        for sigma in cl:
            M = sigma_matrix_on_tensor_product(sigma, n, k)
            Pi += chi * M

    Pi *= d_lam / sage.factorial(k)
    return Pi

In [7]:
n, k = 2, 3  # qubit example: V = C^2, k=3 tensor factors

for lam in sage.Partitions(k).list():
    Pi = isotypic_projector(lam, n, k)
    
    # Check: Pi^2 = Pi (projector)
    assert np.allclose(Pi @ Pi, Pi), f"{lam}: not idempotent"
    
    # Check: Pi is Hermitian
    assert np.allclose(Pi, Pi.conj().T), f"{lam}: not Hermitian"
    
    # Check: trace = d_lambda * multiplicity
    print(f"lambda={lam}, trace={np.trace(Pi).real:.1f}")

# Check: projectors sum to identity
total = sum(isotypic_projector(lam, n, k) 
            for lam in sage.Partitions(k).list())
assert np.allclose(total, np.eye(n**k))
print("Projectors sum to identity ✓")

lambda=[3], trace=4.0
lambda=[2, 1], trace=4.0
lambda=[1, 1, 1], trace=0.0
Projectors sum to identity ✓


In [8]:
import numpy as np
from itertools import product as iproduct

def ket(indices: list[int], n: int) -> np.ndarray:
    """
    Convert a basis vector in ket notation to a vector in (C^n)^{otimes k}.
    
    indices: tuple of ints, e.g. (2, 0, 3) for |2,0,3>
    n: local dimension, e.g. n=4 for C^4
    
    Example: ket((2,0,3), n=4) -> unit vector in C^{4^3} = C^64
    """
    k = len(indices)
    assert all(0 <= i < n for i in indices), f"All indices must be in range [0, {n-1}]"

    basis = list(iproduct(range(n), repeat=k))
    index = {b: i for i, b in enumerate(basis)}

    v = np.zeros(n**k, dtype=complex)
    v[index[tuple(indices)]] = 1.0
    return v

# |2,0,3> in (C^4)^{otimes 3}
v = ket([2,0,3], n=4)
print(v.shape)   # (64,)
print(v.sum())   # 1.0 — exactly one nonzero entry

# Recover the index
print(np.argmax(v))  # 2*16 + 0*4 + 3 = 35

(64,)
(1+0j)
35


In [9]:
v = ket([0, 0, 3], n=4)
Pi = isotypic_projector([3], n=4, k=3)

projected = Pi @ v

# Norm of the projected vector — how much of |2,0,3> lives in this subspace
print(np.linalg.norm(projected))

# Applying the projector twice should give the same result
print(np.allclose(Pi @ projected, projected))  # True

# Projectors onto different sectors are orthogonal
Pi_sym  = isotypic_projector([3],     n=4, k=3)
Pi_mix  = isotypic_projector([2,1],   n=4, k=3)
Pi_anti = isotypic_projector([1,1,1], n=4, k=3)

v_sym  = Pi_sym  @ v
v_mix  = Pi_mix  @ v
v_anti = Pi_anti @ v

print(v_anti)

# These three components are orthogonal and reconstruct v
print(np.allclose(v_sym + v_mix + v_anti, v))           # True
print(np.allclose(np.dot(v_sym.conj(), v_mix), 0))      # True

0.5773502691896257
True
[0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
True
True


In [10]:
def nu(v: list[int], n:int) -> list[int]:
    # n could also be the max instead of a parameter
    ret = [0] * n
    for i in v:
        ret[i] += 1
    ret.sort(reverse=True)
    return [i for i in ret if i != 0]

def maps_to_zero(lam: list[int], n: int, v: list[int], tol: float = 1e-5) -> bool:
    """
    Check if the isotypic projector Pi_lambda maps v to zero.
    
    lam: Young frame, e.g. [3] or [2,1]
    n:   local dimension
    v:   vector in (C^n)^{otimes k}
    tol: numerical tolerance
    """


    k = sum(lam)
    ketv = ket(v, n)
    Pi = isotypic_projector(lam, n, k)
    projected = Pi @ ketv

    print(f"lam = {str(lam)}, v={str(v)}, n={n}, h(lam)={len(lam)}, h(nu)={len(nu(v,n))}, expected={"0" if len(nu(v,n)) < len(lam) else "any"}, res={"0" if np.linalg.norm(projected) < tol else "|*>"}")

    return bool(np.linalg.norm(projected) < tol)

In [11]:
n = 2
v = [0, 0, 1] # some vector in (C^2)^{otimes 3}

print(maps_to_zero([3],     n=n, v=v))  # False — has symmetric component
print(maps_to_zero([1,1,1], n=n, v=v))  # True  — no antisymmetric component (n<k)
print(maps_to_zero([2,1],   n=n, v=v))  # depends on the vector

lam = [3], v=[0, 0, 1], n=2, h(lam)=1, h(nu)=2, expected=any, res=|*>
False
lam = [1, 1, 1], v=[0, 0, 1], n=2, h(lam)=3, h(nu)=2, expected=0, res=0
True
lam = [2, 1], v=[0, 0, 1], n=2, h(lam)=2, h(nu)=2, expected=any, res=|*>
False


In [12]:
n = 3
v = [0, 0, 0, 1, 1, 2]

print(maps_to_zero([6],     n=n, v=v))  # False — has symmetric component
print(maps_to_zero([3,3],     n=n, v=v))  # False — has symmetric component
print(maps_to_zero([2,2,1,1], n=n, v=v))
print(maps_to_zero([2,1,1,1,1], n=n, v=v))
print(maps_to_zero([3,2,1], n=n, v=v))
print(maps_to_zero([4,2],   n=n, v=v))  # depends on the vector

lam = [6], v=[0, 0, 0, 1, 1, 2], n=3, h(lam)=1, h(nu)=3, expected=any, res=|*>
False
lam = [3, 3], v=[0, 0, 0, 1, 1, 2], n=3, h(lam)=2, h(nu)=3, expected=any, res=|*>
False
lam = [2, 2, 1, 1], v=[0, 0, 0, 1, 1, 2], n=3, h(lam)=4, h(nu)=3, expected=0, res=0
True
lam = [2, 1, 1, 1, 1], v=[0, 0, 0, 1, 1, 2], n=3, h(lam)=5, h(nu)=3, expected=0, res=0
True
lam = [3, 2, 1], v=[0, 0, 0, 1, 1, 2], n=3, h(lam)=3, h(nu)=3, expected=any, res=|*>
False
lam = [4, 2], v=[0, 0, 0, 1, 1, 2], n=3, h(lam)=2, h(nu)=3, expected=any, res=|*>
False


In [13]:
def v_cumulative(v: list[int]):
    c = v.copy()
    for i in range(1,len(c)):
        c[i] += c[i-1]
    return c


def majorizes(nuv: list[int], lam: list[int]) -> bool:
    assert sum(lam) == sum(nuv)
    lams = 0
    vs = 0
    for i in range(min(len(lam), len(nuv))):
        lams += lam[i]
        vs += nuv[i]
        print(lams, vs)
        if vs > lams:
            return True
    return False

def maps_to_zero_major(lam: list[int], n: int, v: list[int], tol: float = 1e-5) -> bool:
    """
    Check if the isotypic projector Pi_lambda maps v to zero.
    
    lam: Young frame, e.g. [3] or [2,1]
    n:   local dimension
    v:   vector in (C^n)^{otimes k}
    tol: numerical tolerance
    """


    k = sum(lam)
    ketv = ket(v, n)
    Pi = isotypic_projector(lam, n, k)
    projected = Pi @ ketv

    print(f"lam = {str(lam)}, v={str(v)}, n={n}, lam_c={v_cumulative(lam)}, nu_c={v_cumulative(nu(v,n))}, expected={"0" if majorizes( nu(v,n),lam) else "any"}, res={"0" if np.linalg.norm(projected) < tol else "|*>"}")

    return bool(np.linalg.norm(projected) < tol)

def norm_hook_hard(lambda1 : int, k:int , n: int, v: list[int]):
    ketv = ket(v, n)
    lam: list[int] = [lambda1]
    lam.extend([1] * (k - lambda1))

    print(lam)

    Pi = isotypic_projector(lam, n, k)
    projected = Pi @ ketv
    return np.linalg.norm(projected)



In [14]:
n = 3
v = [0, 0, 0, 1, 1, 2]

print(maps_to_zero_major([6],     n=n, v=v))  # False — has symmetric component
print(maps_to_zero_major([3,3],     n=n, v=v))  # False — has symmetric component
print(maps_to_zero_major([2,2,1,1], n=n, v=v))
print(maps_to_zero_major([2,1,1,1,1], n=n, v=v))
print(maps_to_zero_major([3,2,1], n=n, v=v))
print(maps_to_zero_major([4,2],   n=n, v=v))  # depends on the vector

6 3
lam = [6], v=[0, 0, 0, 1, 1, 2], n=3, lam_c=[6], nu_c=[3, 5, 6], expected=any, res=|*>
False
3 3
6 5
lam = [3, 3], v=[0, 0, 0, 1, 1, 2], n=3, lam_c=[3, 6], nu_c=[3, 5, 6], expected=any, res=|*>
False
2 3
lam = [2, 2, 1, 1], v=[0, 0, 0, 1, 1, 2], n=3, lam_c=[2, 4, 5, 6], nu_c=[3, 5, 6], expected=0, res=0
True
2 3
lam = [2, 1, 1, 1, 1], v=[0, 0, 0, 1, 1, 2], n=3, lam_c=[2, 3, 4, 5, 6], nu_c=[3, 5, 6], expected=0, res=0
True
3 3
5 5
6 6
lam = [3, 2, 1], v=[0, 0, 0, 1, 1, 2], n=3, lam_c=[3, 5, 6], nu_c=[3, 5, 6], expected=any, res=|*>
False
4 3
6 5
lam = [4, 2], v=[0, 0, 0, 1, 1, 2], n=3, lam_c=[4, 6], nu_c=[3, 5, 6], expected=any, res=|*>
False


In [15]:
def norm_hook_fast(lambda1: int, k: int, n: int, v: list[int]) -> float:
    v_nu = nu(v, n)
    numerator = math.comb(k - 1, lambda1 - 1) * math.comb(len(v_nu) - 1, k - lambda1)
    denominator = math.factorial(k) // math.prod(math.factorial(vj) for vj in v_nu)
    return math.sqrt(numerator / denominator)

In [16]:
k = 4
lambda1 = 2
n = 3
v = [0,2,1,1]

print(norm_hook_hard(lambda1, k, n, v))
print(norm_hook_fast(lambda1, k, n, v))

k = 6
lambda1 = 3
n = 4
v = [0,0,1,1,2,3]

# print(norm_hook_hard(lambda1, k, n, v))
print(norm_hook_fast(lambda1, k, n, v))

[2, 1, 1]
0.5
0.5
0.23570226039551584


In [17]:
import numpy as np

def compute_mkh(sis: list[float], degree_k, height_h, h_lambda):
    sisq: list[float] = [si**2 for si in sis]
    mkh_dp = np.zeros((degree_k + 1, height_h + 1))
    mkh_dp[0][0] = 1

    for siq in sisq:
        for d in range(degree_k, 0, -1):
            for h in range(min(d, height_h), 1, -1):
                mkh_dp[d][h] += sum(
                    mkh_dp[r][h-1] * siq**(d-r)
                    for r in range(0, d)
                )
            mkh_dp[d][1] += siq**d

    return mkh_dp[degree_k][h_lambda:] 

In [18]:
def norm_hook_generic(sis: list[float], lambda1: int, k: int):
    sisq: list[float] = [si**2 for si in sis]

    h_lambda = k-lambda1 + 1
    r = len(sisq)

    binom_km1_lambda1m1 = math.comb(k-1, lambda1-1)
    sum_h_to_r = sum([ math.comb(h-1, h_lambda-1) * mkh for h, mkh in zip(range(h_lambda, r+1), compute_mkh(sis, k, r, h_lambda))])

    return binom_km1_lambda1m1 * sum_h_to_r


In [19]:
"""
Tests for norm_hook_generic and compute_mkh.
 
Ground truth: build psi_AB = sum_i s_i |i>_A |i>_B explicitly,
then compute ||Pi_lambda psi_AB^{otimes k}||^2 via isotypic_projector.
 
Run in the Sage environment where the notebook functions are defined.
Paste this file after the notebook cells, or import it.
"""
 
import math
import numpy as np
 
 
# ---------------------------------------------------------------------------
# Helper: build psi_AB^{otimes k} from Schmidt coefficients
# ---------------------------------------------------------------------------
 
def make_psi_AB(sis: list[float]) -> np.ndarray:
    """
    Build the bipartite state psi_AB = sum_i s_i |i>_A |i>_B
    as a vector in C^r otimes C^r, where r = len(sis).
 
    The basis ordering is the standard tensor product ordering:
    |i>_A |j>_B <-> index i*r + j.
    """
    r = len(sis)
    psi = np.zeros(r * r, dtype=complex)
    for i, si in enumerate(sis):
        psi[i * r + i] = si          # coefficient of |i>_A |i>_B
    return psi
 
 
def make_psi_AB_k(sis: list[float], k: int) -> np.ndarray:
    """
    Build psi_AB^{otimes k} as a vector in (C^r otimes C^r)^{otimes k}
    = (C^r)^{otimes k} otimes (C^r)^{otimes k}.
 
    We treat the full local dimension as r^2 (the AB pair as one system
    of dimension r^2) so that psi_AB lives in C^{r^2} and psi_AB^{otimes k}
    lives in (C^{r^2})^{otimes k}.
 
    Equivalently: local dimension n = r^2, and psi_AB is a unit vector
    whose nonzero entries are at positions i*(r+1) for i=0..r-1 (the
    diagonal of the r x r block), with amplitude s_i.
    """
    psi = make_psi_AB(sis)           # vector in C^{r^2}
    psi_k = psi.copy()
    for _ in range(k - 1):
        psi_k = np.kron(psi_k, psi)  # (C^{r^2})^{otimes k}
    return psi_k
 
 
def norm_sq_projected(sis: list[float], lambda1: int, k: int) -> float:
    """
    Ground-truth computation of ||Pi_lambda psi_AB^{otimes k}||^2.
 
    The projector Pi_lambda acts only on the A-subsystem:
    Pi_lambda otimes I_B.
 
    We compute this by:
      1. Building psi_AB^{otimes k} in (C^r otimes C^r)^{otimes k}.
         We treat the AB pair as a single system of dimension n = r^2.
      2. Noting that Pi_lambda otimes I_B on (C^r)^{otimes k} otimes (C^r)^{otimes k}
         is equivalent to (Pi_lambda^{(n=r^2, k)}) where sigma only permutes
         the A-indices and leaves B-indices fixed.
 
    Actually the simplest correct approach: work directly in the A-space.
    psi_AB = sum_i s_i |i>_A |i>_B, so
    psi_AB^{otimes k} = sum_{i1,...,ik} s_{i1}...s_{ik} |i1,...,ik>_A |i1,...,ik>_B.
 
    ||Pi_lambda otimes I_B psi_AB^{otimes k}||^2
      = <psi_AB^{otimes k}| Pi_lambda otimes I_B |psi_AB^{otimes k}>
      = sum_{i1..ik, j1..jk} s_{i..} s_{j..} <i..|_A Pi_lambda |j..>_A  <i..|_B |j..>_B
      = sum_{i1..ik} s_{i1}^2...s_{ik}^2 <i1..ik| Pi_lambda |i1..ik>_A
    because <i..|_B |j..>_B = delta_{i,j} (orthonormal basis).
 
    So we only need Pi_lambda in the A-space (dimension r), applied to
    each basis vector |i1,...,ik> weighted by s_{i1}^2...s_{ik}^2.
    """
    r = len(sis)
    sisq = [si**2 for si in sis]
 
    lam = [lambda1] + [1] * (k - lambda1)   # hook partition
    Pi = isotypic_projector(lam, n=r, k=k)  # acts on (C^r)^{otimes k}
 
    from itertools import product as iproduct
    basis = list(iproduct(range(r), repeat=k))
 
    total = 0.0
    for idx_tuple in basis:
        weight = math.prod(sisq[i] for i in idx_tuple)
        if weight == 0:
            continue
        # <idx_tuple| Pi |idx_tuple> = Pi[row, row] where row = index of idx_tuple
        row = basis.index(idx_tuple)
        total += weight * Pi[row, row].real
 
    return total
 
 
# ---------------------------------------------------------------------------
# Tests
# ---------------------------------------------------------------------------
 
def assert_close(a, b, tol=1e-6, label=""):
    err = abs(a - b)
    status = "PASS" if err < tol else "FAIL"
    print(f"[{status}] {label}")
    if err >= tol:
        print(f"        expected {b}, got {a}, diff {err:.2e}")
 
 
def test_uniform_coefficients():
    """
    Uniform Schmidt coefficients: s_i = 1/sqrt(r).
    All s_i^2 = 1/r so the monomial symmetric functions have known
    combinatorial values: m_nu(1/r,...,1/r) = #{distinct monomials of shape nu} / r^k.
    """
    r = 3
    k = 4
    sis = [1.0 / math.sqrt(r)] * r
 
    for lambda1 in range(1, k + 1):
        formula  = norm_hook_generic(sis, lambda1, k) ** 0.5   # formula gives norm^2, take sqrt for norm... 
        # actually norm_hook_generic returns ||Pi psi||^2, compare directly
        gt = norm_sq_projected(sis, lambda1, k)
        formula_val = norm_hook_generic(sis, lambda1, k)
        assert_close(formula_val, gt,
                     label=f"uniform r={r} k={k} lambda1={lambda1}")
 
 
def test_single_nonzero():
    """
    s = [1, 0, 0]: pure product state, all weight on first Schmidt vector.
    psi_AB = |0>|0>, psi_AB^k = |00...0>.
    The only partition that contributes is nu=(k,), which has h(nu)=1.
    Pi_lambda maps |0...0> to a nonzero vector only if h(lambda)=1,
    i.e. lambda = (lambda1, 1, 1,...,1) which has h(lambda)=k-lambda1+1.
    But |0...0> is symmetric, so Pi_[k] |0...0> = |0...0> and
    Pi_lambda |0...0> = 0 for lambda != [k].
    So norm_hook_generic(sis=[1,0,0], lambda1=k, k=k) should equal 1
    and all others should equal 0.
    """
    r = 3
    k = 3
    sis = [1.0, 0.0, 0.0]
 
    for lambda1 in range(1, k + 1):
        formula_val = norm_hook_generic(sis, lambda1, k)
        gt = norm_sq_projected(sis, lambda1, k)
        assert_close(formula_val, gt,
                     label=f"single-nonzero r={r} k={k} lambda1={lambda1}")
 
 
def test_two_equal():
    """
    s = [1/sqrt(2), 1/sqrt(2), 0]: maximally entangled on 2 dimensions.
    """
    r = 3
    k = 4
    sis = [1.0 / math.sqrt(2), 1.0 / math.sqrt(2), 0.0]
 
    for lambda1 in range(1, k + 1):
        formula_val = norm_hook_generic(sis, lambda1, k)
        gt = norm_sq_projected(sis, lambda1, k)
        assert_close(formula_val, gt,
                     label=f"two-equal r={r} k={k} lambda1={lambda1}")
 
 
def test_generic_coefficients():
    """
    Generic Schmidt coefficients (not uniform, not sparse).
    Compare formula against ground truth for several (k, lambda1) pairs.
    """
    sis = [0.7, 0.5, 0.3, 0.2]
    # normalise so sum(si^2) = 1
    norm = math.sqrt(sum(s**2 for s in sis))
    sis = [s / norm for s in sis]
 
    for k in [2, 3, 4]:
        for lambda1 in range(1, k + 1):
            formula_val = norm_hook_generic(sis, lambda1, k)
            gt = norm_sq_projected(sis, lambda1, k)
            assert_close(formula_val, gt,
                         label=f"generic sis k={k} lambda1={lambda1}")
 
 
def test_norm_sq_sums_to_one():
    """
    Sum over all hook partitions lambda=(lambda1,1,...,1) of ||Pi_lambda psi||^2
    should equal ||psi||^2 = 1, since the hook partitions do NOT exhaust all
    of S_k. So this tests something weaker: the formula is internally consistent
    when summed with the non-hook projectors.
 
    Instead, we check: sum over ALL partitions of k of norm_sq = 1,
    using norm_sq_projected as ground truth.
    """
    r = 3
    k = 3
    sis = [0.8, 0.5, 0.2]
    norm = math.sqrt(sum(s**2 for s in sis))
    sis = [s / norm for s in sis]
 
    # sum over all hook partitions lambda1=1..k
    total = sum(norm_sq_projected(sis, lambda1, k) for lambda1 in range(1, k + 1))
    # hooks don't cover everything for k>=3; sum should be <= 1
    # Instead check each individually matches formula
    for lambda1 in range(1, k + 1):
        formula_val = norm_hook_generic(sis, lambda1, k)
        gt = norm_sq_projected(sis, lambda1, k)
        assert_close(formula_val, gt,
                     label=f"sum-consistency r={r} k={k} lambda1={lambda1}")
 
 
def test_compute_mkh_special_cases():
    """
    Unit tests for compute_mkh directly.
 
    - h=1: m_k^(1) = sum_i si^{2k} = power sum p_k(sisq)
    - h=r: m_k^(r) = h_k(sisq) - sum_{h<r} m_k^(h)  (unconstrained)
    - All entries nonneg (monomials with positive coefficients)
    """
    sis = [0.6, 0.5, 0.4]
    sisq = [s**2 for s in sis]
    k = 4
    r = len(sis)
 
    mkh_all = compute_mkh(sis, k, r, 1)   # h_lambda=1, returns h=1..r
 
    # h=1 should be sum si^{2k}
    expected_h1 = sum(sq**k for sq in sisq)
    assert_close(float(mkh_all[0]), expected_h1,
                 label=f"compute_mkh h=1 equals p_k(sisq)")
 
    # All entries should be nonneg
    for h_idx, val in enumerate(mkh_all):
        h = h_idx + 1
        ok = float(val) >= -1e-10
        status = "PASS" if ok else "FAIL"
        print(f"[{status}] compute_mkh nonneg at h={h}, val={float(val):.6f}")
 
    # Sum of m_k^(h) for h=1..r should equal h_k(sisq) (unconstrained)
    from fractions import Fraction
    # compute h_k via Newton recurrence
    s_pows = {j: sum(sq**j for sq in sisq) for j in range(1, k+1)}
    hh = {0: 1.0}
    for m in range(1, k+1):
        hh[m] = sum(s_pows[j] * hh[m-j] for j in range(1, m+1)) / m
    expected_hk = hh[k]
    assert_close(float(sum(mkh_all)), expected_hk,
                 label=f"sum of m_k^(h) over h=1..r equals h_k(sisq)")
 
 
# ---------------------------------------------------------------------------
# Run all tests
# ---------------------------------------------------------------------------
 
def run_all_tests():
    print("=" * 60)
    print("Running tests for norm_hook_generic and compute_mkh")
    print("=" * 60)
    test_compute_mkh_special_cases()
    test_uniform_coefficients()
    test_single_nonzero()
    test_two_equal()
    test_generic_coefficients()
    test_norm_sq_sums_to_one()
    print("=" * 60)
    print("Done.")
 
run_all_tests()

Running tests for norm_hook_generic and compute_mkh
[PASS] compute_mkh h=1 equals p_k(sisq)
[PASS] compute_mkh nonneg at h=1, val=0.021358
[PASS] compute_mkh nonneg at h=2, val=0.042770
[PASS] compute_mkh nonneg at h=3, val=0.011088
[PASS] sum of m_k^(h) over h=1..r equals h_k(sisq)
[PASS] uniform r=3 k=4 lambda1=1
[PASS] uniform r=3 k=4 lambda1=2
[PASS] uniform r=3 k=4 lambda1=3
[PASS] uniform r=3 k=4 lambda1=4
[PASS] single-nonzero r=3 k=3 lambda1=1
[PASS] single-nonzero r=3 k=3 lambda1=2
[PASS] single-nonzero r=3 k=3 lambda1=3
[PASS] two-equal r=3 k=4 lambda1=1
[PASS] two-equal r=3 k=4 lambda1=2
[PASS] two-equal r=3 k=4 lambda1=3
[PASS] two-equal r=3 k=4 lambda1=4
[PASS] generic sis k=2 lambda1=1
[PASS] generic sis k=2 lambda1=2
[PASS] generic sis k=3 lambda1=1
[PASS] generic sis k=3 lambda1=2
[PASS] generic sis k=3 lambda1=3
[PASS] generic sis k=4 lambda1=1
[PASS] generic sis k=4 lambda1=2
[PASS] generic sis k=4 lambda1=3
[PASS] generic sis k=4 lambda1=4
[PASS] sum-consistency r=3 

## Schur polynomial formula

$$\|\Pi_\lambda |\psi_{AB}\rangle^{\otimes k}\|^2 = f^\lambda \cdot s_\lambda(s_1^2, s_2^2, \ldots, s_r^2)$$

In [20]:
def f_lambda(lam: list[int]) -> int:
    """Number of standard Young tableaux of shape lam, via the hook-length formula f^lambda = k! / prod h(u)."""
    lam = sorted(lam, reverse=True)
    k = sum(lam)
    hook_product = 1
    for i, row_len in enumerate(lam):
        for j in range(row_len):
            arm = row_len - j - 1                          # cells to the right
            leg = sum(1 for r in lam[i + 1:] if r > j)    # cells below
            hook_product *= arm + leg + 1
    return math.factorial(k) // hook_product

# Sanity checks against known values
assert f_lambda([3])     == 1   # single row
assert f_lambda([1,1,1]) == 1   # single column
assert f_lambda([2,1])   == 2
assert f_lambda([3,2,1]) == 16
print("f_lambda checks passed ✓")

f_lambda checks passed ✓


In [21]:
def norm_schur(lam: list[int], sis: list[float]) -> float:
    """||Pi_lambda psi_AB^{otimes k}||^2 = f^lambda * s_lambda(s1^2, ..., sr^2).

    lam: Young frame partition (any order, will be sorted)
    sis: Schmidt coefficients with sum(si^2) = 1
    """
    sisq = [si**2 for si in sis]
    return f_lambda(lam) * schur_polynomial(lam, sisq)


# Quick checks for schur_polynomial
# s_{[1]}(x) = x_1 + x_2  →  sum = 1
# Patch: allow Python floats in evaluation by working over RDF
def schur_polynomial(lam: list[int], x: list[float]) -> float:
    lam = sorted(lam, reverse=True)
    r = len(x)
    poly = _s[lam].expand(r).change_ring(sage.RDF)
    return float(poly(*[sage.RDF(xi) for xi in x]))

assert abs(schur_polynomial([1], [0.3, 0.7]) - 1.0) < 1e-10

# s_{[2]} = h_2: 0.25 + 0.25 + 0.25 = 0.75 at (0.5, 0.5)
assert abs(schur_polynomial([2], [0.5, 0.5]) - 0.75) < 1e-10

# s_{[1,1]} = e_2: 0.5 * 0.5 = 0.25 at (0.5, 0.5)
assert abs(schur_polynomial([1, 1], [0.5, 0.5]) - 0.25) < 1e-10

# height > r  →  0
assert abs(schur_polynomial([1, 1, 1], [0.5, 0.5])) < 1e-10

print("schur_polynomial checks passed ✓")

schur_polynomial checks passed ✓


In [22]:
def norm_sq_projected_general(lam: list[int], sis: list[float]) -> float:
    """Ground-truth ||Pi_lam psi_AB^{otimes k}||^2 for any partition lam."""
    r = len(sis)
    sisq = [si**2 for si in sis]
    k = sum(lam)
    Pi = isotypic_projector(lam, n=r, k=k)
    basis = list(iproduct(range(r), repeat=k))
    basis_index = {b: i for i, b in enumerate(basis)}
    total = 0.0
    for idx_tuple in basis:
        weight = math.prod(sisq[i] for i in idx_tuple)
        if weight == 0:
            continue
        total += weight * Pi[basis_index[idx_tuple], basis_index[idx_tuple]].real
    return total


def assert_close(a, b, tol=1e-6, label=""):
    err = abs(a - b)
    status = "PASS" if err < tol else "FAIL"
    print(f"[{status}] {label}")
    if err >= tol:
        print(f"        got {a:.8f}, expected {b:.8f}, diff {err:.2e}")


# --- Test 1: hook partitions ---
sis = [0.7, 0.5, 0.3]
sis = [s / math.sqrt(sum(x**2 for x in sis)) for s in sis]
k = 4
print("Hook partitions (k=4, r=3):")
for lambda1 in range(1, k + 1):
    lam = [lambda1] + [1] * (k - lambda1)
    assert_close(norm_schur(lam, sis), norm_sq_projected(sis, lambda1, k), label=f"lam={lam}")

# --- Test 2: non-hook partitions ---
sis2 = [0.8, 0.5, 0.3, 0.1]
sis2 = [s / math.sqrt(sum(x**2 for x in sis2)) for s in sis2]
print("\nNon-hook partitions (r=4):")
for lam in [[2, 2], [3, 1], [2, 1, 1], [2, 2, 1]]:
    assert_close(norm_schur(lam, sis2), norm_sq_projected_general(lam, sis2), label=f"lam={lam}")

# --- Test 3: sum over all partitions of k equals 1 ---
k = 3
sis3 = [1/math.sqrt(2), 1/math.sqrt(3), 1/math.sqrt(6)]
total = sum(norm_schur(list(lam), sis3) for lam in sage.Partitions(k).list())
assert_close(total, 1.0, label=f"sum over all partitions of {k} equals 1")

Hook partitions (k=4, r=3):
[PASS] lam=[1, 1, 1, 1]
[PASS] lam=[2, 1, 1]
[PASS] lam=[3, 1]
[PASS] lam=[4]

Non-hook partitions (r=4):
[PASS] lam=[2, 2]
[PASS] lam=[3, 1]
[PASS] lam=[2, 1, 1]
[PASS] lam=[2, 2, 1]
[PASS] sum over all partitions of 3 equals 1


## Schmidt-number SDP (symmetric-extension relaxation)

The pure-state result above, $ | (\Pi^\lambda_A\otimes I_B) | \psi_{AB} \rangle^{ \otimes k} | ^2=f^\lambda s_\lambda(s_1^2,\dots,s_r^2)$,
vanishes **iff** $\mathrm{SR}(\psi)<h(\lambda)$. Extending to mixed $\rho$ via the convex roof gives, exactly,

$$\mathrm{SN}(\rho)<h(\lambda)\;\iff\;\min_{\omega}\ \mathrm{tr}\!\big[(\Pi^\lambda_A\otimes I_B)\,\omega\big]=0,$$

the minimum over $\omega=\sum_i p_i(|\psi_i\rangle\langle\psi_i|)^{\otimes k}$ with $\mathrm{tr}_{2\dots k}\omega=\rho$.
That set (convex hull of identical pure powers) is intractable, so we **relax** it to the
Tóth–Moroder–Gühne symmetric extension: $\omega\succeq0$, supported on $\mathrm{Sym}^k$,
with the right one-copy marginal, and PPT across the copy cuts. The relaxed set is a *superset*,
so the SDP optimum **lower-bounds** the convex roof:

$$\boxed{\;v^\star_\lambda>0\ \Longrightarrow\ \mathrm{SN}(\rho)\ge h(\lambda)\;}\qquad(v^\star_\lambda=0:\ \text{inconclusive}).$$

We work in the ordering $A_1\cdots A_k\,B_1\cdots B_k$, so the objective is just
$W=\Pi^\lambda_A\otimes\I_{B^{\otimes k}}$ (a Kronecker product), while marginal/PPT act on the
$2k$ subsystems $[\,d_A,\dots,d_A,d_B,\dots,d_B\,]$. A copy is one $AB$ pair; a copy-permutation
acts *simultaneously* on the matching $A_j$ and $B_j$, so $V_\pi=P^A_\pi\otimes P^B_\pi$ and
$P_{\mathrm{sym}}=\frac1{k!}\sum_\pi V_\pi$.

In [ ]:
# One-time: install the SDP stack into the Sage Python environment.
# (Run once; restart the kernel afterwards if the import below fails.)
import sys
get_ipython().system(f"{sys.executable} -m pip install --quiet picos cvxopt")

In [ ]:
import numpy as np
import picos as pic

def bose_projector(dA: int, dB: int, k: int) -> np.ndarray:
    """P_sym onto Sym^k of the k AB-copies, ordering A_1..A_k B_1..B_k.
    A copy-permutation acts jointly on the A-factors and the B-factors:
    V_pi = P^A_pi (x) P^B_pi. Reuses sigma_matrix_on_tensor_product."""
    Sk = sage.SymmetricGroup(k)
    D = (dA**k) * (dB**k)
    Psym = np.zeros((D, D), dtype=complex)
    for sigma in Sk:
        PA = sigma_matrix_on_tensor_product(sigma, dA, k)
        PB = sigma_matrix_on_tensor_product(sigma, dB, k)
        Psym += np.kron(PA, PB)
    Psym /= float(sage.factorial(k))
    return Psym


def schmidt_number_sdp(rho, lam: list[int], dA: int, dB: int,
                       ppt: bool = True, solver=None, tol: float = 1e-7):
    """Symmetric-extension SDP lower bound on the convex-roof value
           min_decomp  sum_i p_i || (Pi^lam_A (x) I_B) |psi_i>^{(x)k} ||^2 .

    rho : density matrix on C^{dA} (x) C^{dB}, ordering |a>|b> -> a*dB + b.
    lam : partition of k = sum(lam); the test certifies SN(rho) >= h(lam) = len(lam).

    Returns (value, certified) where certified == (value > tol).
    value > tol  =>  SN(rho) >= h(lam).   value ~ 0  =>  inconclusive.
    """
    lam = sorted(lam, reverse=True)
    k = sum(lam)
    D = dA * dB

    PiA = isotypic_projector(lam, dA, k)              # Pi^lam on (C^dA)^{(x)k}
    W   = np.kron(PiA, np.eye(dB**k))                 # objective, ordering A_1..A_k B_1..B_k
    Psym = bose_projector(dA, dB, k)

    dims = [dA]*k + [dB]*k                            # 2k subsystems
    keep = [0, k]                                     # A_1 and B_1  (one full AB copy)
    trace_out = [i for i in range(2*k) if i not in keep]

    P = pic.Problem()
    w = pic.HermitianVariable("w", (D**k, D**k))
    P.add_constraint(w >> 0)                                                  # PSD
    P.add_constraint(pic.Constant(Psym) * w * pic.Constant(Psym) == w)       # supported on Sym^k
    marg = pic.partial_trace(w, subsystems=trace_out, dimensions=dims)
    P.add_constraint(marg == pic.Constant(np.asarray(rho, dtype=complex)))    # tr_{2..k} w = rho
    if ppt:
        for r in range(1, k//2 + 1):                 # representative copy-cuts |S| = 1..floor(k/2)
            S = list(range(r))
            sub = S + [k + j for j in S]             # transpose A_j and B_j for j in S
            P.add_constraint(pic.partial_transpose(w, subsystems=sub, dimensions=dims) >> 0)

    P.set_objective("min", (pic.Constant(W) | w).real)   # <W, w> = tr(W w)
    P.solve(solver=solver, primals=True)
    val = float(P.value)
    return val, bool(val > tol)

In [ ]:
# Benchmark: d x d isotropic states.  For lam = (1,...,1) of length t,
# the test certifies SN >= t.  Known threshold: SN >= t  <=>  fidelity F > (t-1)/d.
# Here d = 2, t = 2 :  SN >= 2 (entangled)  <=>  F > 1/2  (p > 1/3).

def isotropic(d: int, p: float):
    phi = np.zeros(d*d, dtype=complex)
    for i in range(d):
        phi[i*d + i] = 1/np.sqrt(d)
    Phi = np.outer(phi, phi.conj())
    rho = p*Phi + (1-p)*np.eye(d*d)/(d*d)
    return rho, phi

d = 2
print(f"{'p':>5} {'F':>6} {'SDP value':>12}   certify SN>=2 ?")
for p in [0.0, 0.20, 1/3, 0.34, 0.50, 0.80, 1.0]:
    rho, phi = isotropic(d, p)
    val, cert = schmidt_number_sdp(rho, [1, 1], dA=d, dB=d)
    F = float((phi.conj() @ rho @ phi).real)
    print(f"{p:5.2f} {F:6.3f} {val:12.3e}   {'YES' if cert else 'no'}")

print("\nExpected: certified exactly for F > 1/2  (p > 1/3).")

### Reading the output and scaling up

- **One-sided certificate.** A strictly positive optimum proves $\mathrm{SN}(\rho)\ge h(\lambda)$.
  A zero is *inconclusive* — it does not prove low Schmidt number, because the relaxed
  $\omega$ need not be a genuine mixture of identical pure powers (that gap is exactly the
  separability problem).
- **Which $(\lambda,k)$ to pick.** $h(\lambda)$ is the Schmidt number you test. At fixed $k$,
  the *shape* of $\lambda$ only changes the objective $W$ (same feasible set), so you can sweep
  all $\lambda\vdash k$ cheaply after building $P_{\mathrm{sym}}$ once. Taking $\lambda$ of the
  same height but **larger $k$** enlarges $\mathrm{Sym}^k$ and tightens the bound — a de Finetti
  hierarchy, monotonically stronger but more expensive. The single-column $\lambda=(1^t)$
  (antisymmetrizer) is the minimal, cleanest detector of $\mathrm{SN}\ge t$.
- **Cost.** The variable here is $D^k\times D^k$ with $D=d_Ad_B$, so it grows fast. To go beyond
  small cases: (i) restrict $\omega$ to the symmetric subspace via an isometry
  $V:\mathrm{Sym}^k(\mathbb C^D)\hookrightarrow(\mathbb C^D)^{\otimes k}$ and optimize over the
  $\binom{D+k-1}{k}\times\binom{D+k-1}{k}$ matrix $\sigma$ with $\omega=V\sigma V^\dagger$;
  and (ii) use a large-scale conic solver (MOSEK or SCS) instead of cvxopt.
- **Next experiment.** Sweep $d=3,k=2$ (threshold $F>1/3$ for $\mathrm{SN}\ge2$) and
  $d=3$, $\lambda=(1,1,1)$ (threshold $F>2/3$ for $\mathrm{SN}\ge3$); then check whether any
  *non-hook* $\lambda$ at fixed $k$ certifies states that the antisymmetric choice misses —
  that is where this construction could beat existing moment / $k$-reduction witnesses.